## Here we are forming categories for
1. Annual generation potential for each building
2. Household population
3. EV infrastructure

each as High, moderate and low (from the final data set) based on box plot

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd

# Path to your dataset in Google Drive
file_path = "/content/drive/MyDrive/Asset Mapping Project/Final content/Copy of Restarting_Energy_Project/DFW_Assets_Master (3).csv"

# Load dataset
df = pd.read_csv(file_path)

# Preview dataset
print(df.head())

# Check total rows and columns
print("\nDataset Shape:")
print(df.shape)

# Show all column names
print("\nColumns:")
print(df.columns.tolist())

                                     name    building                county  \
0                        NorthPark Center      retail  Dallas County, Texas   
1               Industrial on Lance Drive  industrial  Dallas County, Texas   
2  Commercial on North Central Expressway  commercial  Dallas County, Texas   
3              Unbridled Living of Dallas  commercial  Dallas County, Texas   
4                Office on Hillcrest Road      office  Dallas County, Texas   

             Nearest_Street   latitude  longitude  Building_Sq_Ft  Roof_Sq_Ft  \
0  North Central Expressway  32.868393 -96.773500    1.219857e+06   792906.86   
1               Lance Drive  32.717254 -97.030512    1.206589e+06   784282.60   
2  North Central Expressway  32.979636 -96.715736    5.527943e+04    35931.63   
3                 Coit Road  32.916452 -96.770310    1.151029e+05    74816.88   
4            Hillcrest Road  32.923485 -96.785250    5.662391e+04    36805.54   

      Roof_Sq_M  Annual_kWh  Annual_MW

## Interactive Distribution Visualization

To visualize the distributions of `Annual_MWh`, `Households_1mi`, and `EV_Stations_1mi` interactively, we will use Plotly. Interactive histograms provide a dynamic way to explore the data, allowing for zooming, panning, and viewing precise values.

In [3]:
import plotly.express as px

# =========================================================
# VISUALIZE DATA DISTRIBUTION (Interactive Plotly)
# =========================================================

columns_to_plot = [
    'Annual_MWh',
    'Households_1mi',
    'EV_Stations_1mi'
]

for col in columns_to_plot:
    # Interactive Histogram
    fig_hist = px.histogram(df, x=col, nbins=50, title=f'Interactive Distribution of {col}')
    fig_hist.show()

    # Interactive Boxplot (Horizontal)
    fig_box = px.box(df, x=col, orientation='h', title=f'Interactive Box Plot of {col}')
    fig_box.show()

## classification based on Box Plot Statistics

classifying `Annual_MWh`, `Households_1mi`, and `EV_Stations_1mi` based on their box plot characteristics:

*   **Low**: Values from the minimum up to and including the median.
*   **Medium**: Values strictly above the median up to and including the upper fence (Q3 + 1.5 * IQR).
*   **High**: Values strictly above the upper fence.

New classification columns will be created (e.g., `Annual_Class_BP`) to distinguish them from previous classifications.

In [4]:
# =========================================================
# STEP 3 — Create High / Medium / Low classifications (Box Plot based - NEW)
# =========================================================

# --- Annual Generation Potential ---
annual_median_bp = df['Annual_MWh'].median()
annual_Q1_bp = df['Annual_MWh'].quantile(0.25)
annual_Q3_bp = df['Annual_MWh'].quantile(0.75)
annual_IQR_bp = annual_Q3_bp - annual_Q1_bp
annual_upper_fence_bp = annual_Q3_bp + 1.5 * annual_IQR_bp

def classify_annual_new_bp(x):
    if x <= annual_median_bp:
        return "Low"
    elif x <= annual_upper_fence_bp:
        return "Medium"
    else:
        return "High"

df['Annual_Class_BP'] = df['Annual_MWh'].apply(classify_annual_new_bp)

# --- Household Population ---
hh_median_bp = df['Households_1mi'].median()
hh_Q1_bp = df['Households_1mi'].quantile(0.25)
hh_Q3_bp = df['Households_1mi'].quantile(0.75)
hh_IQR_bp = hh_Q3_bp - hh_Q1_bp
hh_upper_fence_bp = hh_Q3_bp + 1.5 * hh_IQR_bp

def classify_households_new_bp(x):
    if x <= hh_median_bp:
        return "Low"
    elif x <= hh_upper_fence_bp:
        return "Medium"
    else:
        return "High"

df['Household_Class_BP'] = df['Households_1mi'].apply(classify_households_new_bp)

# --- EV Infrastructure ---
ev_median_bp = df['EV_Stations_1mi'].median()
ev_Q1_bp = df['EV_Stations_1mi'].quantile(0.25)
ev_Q3_bp = df['EV_Stations_1mi'].quantile(0.75)
ev_IQR_bp = ev_Q3_bp - ev_Q1_bp
ev_upper_fence_bp = ev_Q3_bp + 1.5 * ev_IQR_bp

def classify_ev_new_bp(x):
    if x <= ev_median_bp:
        return "Low"
    elif x <= ev_upper_fence_bp:
        return "Medium"
    else:
        return "High"

df['EV_Class_BP'] = df['EV_Stations_1mi'].apply(classify_ev_new_bp)


print("New classification complete using box plot based thresholds.")

New classification complete using box plot based thresholds.


## Summary of New Box Plot Based Classification

In [5]:
classification_summary_bp = []

columns_to_analyze_bp = {
    'Annual_MWh': 'Annual_Class_BP',
    'Households_1mi': 'Household_Class_BP',
    'EV_Stations_1mi': 'EV_Class_BP'
}

for original_col, class_col in columns_to_analyze_bp.items():
    for class_name in ['Low', 'Medium', 'High']:
        # Filter data for the current class
        class_data = df[df[class_col] == class_name][original_col]

        if not class_data.empty:
            count = len(class_data)
            min_val = class_data.min()
            max_val = class_data.max()
            classification_summary_bp.append({
                'Metric': original_col,
                'Class': class_name,
                'Count': count,
                'Min Value': f'{min_val:.2f}',
                'Max Value': f'{max_val:.2f}'
            })
        else:
            classification_summary_bp.append({
                'Metric': original_col,
                'Class': class_name,
                'Count': 0,
                'Min Value': 'N/A',
                'Max Value': 'N/A'
            })

summary_df_bp = pd.DataFrame(classification_summary_bp)
print("Summary of box plot based classification:")
display(summary_df_bp)

print("\nPreview of DataFrame with new classification columns:")
display(df[['Annual_MWh', 'Annual_Class_BP',
            'Households_1mi', 'Household_Class_BP',
            'EV_Stations_1mi', 'EV_Class_BP']].head())

Summary of box plot based classification:


,Metric,Class,Count,Min Value,Max Value
0,Annual_MWh,Low,4306,388.00,821.59
1,Annual_MWh,Medium,3218,821.82,3168.87
2,Annual_MWh,High,1088,3174.26,97104.95
3,Households_1mi,Low,4307,0.00,3911.00
4,Households_1mi,Medium,3841,3912.00,10687.00
5,Households_1mi,High,464,10692.00,26836.00
6,EV_Stations_1mi,Low,4893,0.00,2.00
7,EV_Stations_1mi,Medium,2995,3.00,12.00
8,EV_Stations_1mi,High,724,13.00,86.00



Preview of DataFrame with new classification columns:


,Annual_MWh,Annual_Class_BP,Households_1mi,Household_Class_BP,EV_Stations_1mi,EV_Class_BP
0,23659.93,High,11474,High,4,Medium
1,23402.58,High,2673,Low,1,Low
2,1072.18,Medium,4517,Medium,11,Medium
3,2232.50,Medium,6819,Medium,5,Medium
4,1098.26,Medium,4719,Medium,7,Medium


## Verify Added Classification Columns

The `Annual_Class`, `Household_Class`, and `EV_Class` columns have been added to your primary DataFrame (`df`). This section confirms their presence by displaying the head of the DataFrame, focusing on the original and newly created classification columns.

In [6]:
display(df[['Annual_MWh', 'Annual_Class_BP',
            'Households_1mi', 'Household_Class_BP',
            'EV_Stations_1mi', 'EV_Class_BP']].head())

,Annual_MWh,Annual_Class_BP,Households_1mi,Household_Class_BP,EV_Stations_1mi,EV_Class_BP
0,23659.93,High,11474,High,4,Medium
1,23402.58,High,2673,Low,1,Low
2,1072.18,Medium,4517,Medium,11,Medium
3,2232.50,Medium,6819,Medium,5,Medium
4,1098.26,Medium,4719,Medium,7,Medium


## Download Processed Dataset

To download the DataFrame `df` with the newly added classification columns, we can save it as a CSV file to Google Drive. This makes the processed data easily accessible for further use or local download.

In [7]:
# Define the path where you want to save the CSV file in Google Drive
output_file_path = "/content/drive/MyDrive/Asset Mapping Project/Processed_DFW_Assets_Master.csv"

# Save the DataFrame to a CSV file
df.to_csv(output_file_path, index=False)

print(f"Dataset successfully saved to: {output_file_path}")

Dataset successfully saved to: /content/drive/MyDrive/Asset Mapping Project/Processed_DFW_Assets_Master.csv
